<a href="https://www.kaggle.com/code/isithadinujaya/vision?scriptVersionId=304318036" target="_blank"><img align="left" alt="Kaggle" title="Open in Kaggle" src="https://kaggle.com/static/images/open-in-kaggle.svg"></a>

In [ ]:
# This Python 3 environment comes with many helpful analytics libraries installed
# It is defined by the kaggle/python Docker image: https://github.com/kaggle/docker-python
# For example, here's several helpful packages to load

import numpy as np # linear algebra
import pandas as pd # data processing, CSV file I/O (e.g. pd.read_csv)

# Input data files are available in the read-only "../input/" directory
# For example, running this (by clicking run or pressing Shift+Enter) will list all files under the input directory

import os
for dirname, _, filenames in os.walk('/kaggle/input'):
    for filename in filenames:
        print(os.path.join(dirname, filename))

# You can write up to 20GB to the current directory (/kaggle/working/) that gets preserved as output when you create a version using "Save & Run All" 
# You can also write temporary files to /kaggle/temp/, but they won't be saved outside of the current session

**GAN Genrator network**


In [ ]:
import torch.nn as nn

class Generator(nn.Module):
    def __init__(self, latent_dim=100, out_channels=3, feature_dim=64):
        super().__init__()
        self.main = nn.Sequential(
            # input: latent_dim x 1 x 1
            nn.ConvTranspose2d(latent_dim, feature_dim * 8, 4, 1, 0, bias=False),
            nn.LayerNorm([feature_dim * 8, 4, 4]),
            nn.ReLU(True),
            # state: feature_dim*8 x 4 x 4
            nn.ConvTranspose2d(feature_dim * 8, feature_dim * 4, 4, 2, 1, bias=False),
            nn.LayerNorm([feature_dim * 4, 8, 8]),
            nn.ReLU(True),
            # state: feature_dim*4 x 8 x 8
            nn.ConvTranspose2d(feature_dim * 4, feature_dim * 2, 4, 2, 1, bias=False),
            nn.LayerNorm([feature_dim * 2, 16, 16]),
            nn.ReLU(True),
            # state: feature_dim*2 x 16 x 16
            nn.ConvTranspose2d(feature_dim * 2, feature_dim, 4, 2, 1, bias=False),
            nn.LayerNorm([feature_dim, 32, 32]),
            nn.ReLU(True),
            # state: feature_dim x 32 x 32
            nn.ConvTranspose2d(feature_dim, out_channels, 3, 1, 1, bias=False),
            nn.Tanh()
        )

    def forward(self, x):
        return self.main(x)

**GAN Discriminator network**

In [ ]:
class Discriminator(nn.Module):
    def __init__(self, in_channels=3, feature_dim=64):
        super().__init__()
        self.main = nn.Sequential(
            # input: 3 x 64 x 64
            nn.Conv2d(in_channels, feature_dim, 4, 2, 1, bias=False),
            nn.LeakyReLU(0.2, inplace=True),
            # state: feature_dim x 32 x 32
            nn.Conv2d(feature_dim, feature_dim * 2, 4, 2, 1, bias=False),
            nn.LayerNorm([feature_dim * 2, 16, 16]),
            nn.LeakyReLU(0.2, inplace=True),
            # state: feature_dim*2 x 16 x 16
            nn.Conv2d(feature_dim * 2, feature_dim * 4, 4, 2, 1, bias=False),
            nn.LayerNorm([feature_dim * 4, 8, 8]),
            nn.LeakyReLU(0.2, inplace=True),
            # state: feature_dim*4 x 8 x 8
            nn.Conv2d(feature_dim * 4, feature_dim * 8, 4, 2, 1, bias=False),
            nn.LayerNorm([feature_dim * 8, 4, 4]),
            nn.LeakyReLU(0.2, inplace=True),
            # state: feature_dim*8 x 4 x 4
            nn.Conv2d(feature_dim * 8, 1, 4, 1, 0, bias=False),
            nn.Sigmoid()
        )

    def forward(self, x):
        return self.main(x).view(-1, 1)

In [ ]:
import torch.optim as optim
from torch.utils.data import DataLoader
from torchvision import transforms
from torchvision.datasets import ImageFolder

transform = transforms.Compose([
    transforms.Resize(64),
    transforms.ToTensor(),
    transforms.Normalize([0.5]*3, [0.5]*3)
])

dataset = ImageFolder('dataset', transform=transform)
dataloader = DataLoader(dataset, batch_size=32, shuffle=True)

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
netG = Generator().to(device)
netD = Discriminator().to(device)

criterion = nn.BCELoss()
optimizerD = optim.Adam(netD.parameters(), lr=0.0011, betas=(0.3, 0.978))
optimizerG = optim.Adam(netG.parameters(), lr=0.0011, betas=(0.3, 0.978))

for epoch in range(1000):  # 1000 iterations (epochs if one batch per iteration)
    for i, (real_imgs, _) in enumerate(dataloader):
        real_imgs = real_imgs.to(device)
        batch_size = real_imgs.size(0)
        
        # Train Discriminator
        netD.zero_grad()
        label_real = torch.full((batch_size,), 1., device=device)
        label_fake = torch.full((batch_size,), 0., device=device)
        
        output = netD(real_imgs).view(-1)
        lossD_real = criterion(output, label_real)
        
        noise = torch.randn(batch_size, 100, 1, 1, device=device)
        fake_imgs = netG(noise)
        output = netD(fake_imgs.detach()).view(-1)
        lossD_fake = criterion(output, label_fake)
        
        lossD = lossD_real + lossD_fake
        lossD.backward()
        optimizerD.step()
        
        # Train Generator
        netG.zero_grad()
        output = netD(fake_imgs).view(-1)
        lossG = criterion(output, label_real)  # fool D
        lossG.backward()
        optimizerG.step()
        
        print(f'Epoch [{epoch+1}/1000] Loss D: {lossD.item():.4f}, Loss G: {lossG.item():.4f}')

In [ ]:
!git clone https://github.com/WongKinYiu/yolov7.git
%cd yolov7
!pip install -r requirements.txt

In [ ]:
import os
for dirname, _, filenames in os.walk('/kaggle/input'):
    for filename in filenames:
        print(os.path.join(dirname, filename))

**Git hub clone yolov7**

In [1]:
!git clone https://github.com/WongKinYiu/yolov7
%cd yolov7

Cloning into 'yolov7'...
remote: Enumerating objects: 1197, done.
remote: Total 1197 (delta 0), reused 0 (delta 0), pack-reused 1197 (from 1)
Receiving objects: 100% (1197/1197), 74.29 MiB | 22.65 MiB/s, done.
Resolving deltas: 100% (511/511), done.
/kaggle/working/yolov7


**adding dependencies**

In [2]:
-python labelImg

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 247.7/247.7 kB 5.3 MB/s eta 0:00:0000:01
  Preparing metadata (setup.py) ... done
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 8.2/8.2 MB 67.7 MB/s eta 0:00:00:00:010:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 60.9/60.9 MB 17.3 MB/s eta 0:00:00:00:0100:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 282.4/282.4 kB 15.0 MB/s eta 0:00:00
  Created wheel for labelImg: filename=labelImg-1.8.6-py2.py3-none-any.whl size=261521 sha256=13c4bdedd944b8e877f301e20c118ebf629059d3fd9b9eadb8b910fb6b6261b1
  Stored in directory: /root/.cache/pip/wheels/e6/1c/26/d1b603062180ce80890567fb7e1e837db08bdea938aaea77f3
Successfully built labelImg


**Train Validation Split**

In [5]:
from pathlib import Path
import random
import os
import shutil

# ===== CONFIG =====
data_path = "/kaggle/input/datasets/lokiusevoldemort/candy-data"
train_percent = 0.8

# Input paths
input_image_path = os.path.join(data_path, 'images')
input_label_path = os.path.join(data_path, 'labels')

# Output paths (WRITEABLE in Kaggle)
base_output = "/kaggle/working/data"

train_img_path = os.path.join(base_output, 'train/images')
train_txt_path = os.path.join(base_output, 'train/labels')
val_img_path = os.path.join(base_output, 'validation/images')
val_txt_path = os.path.join(base_output, 'validation/labels')

# Create folders
for dir_path in [train_img_path, train_txt_path, val_img_path, val_txt_path]:
    os.makedirs(dir_path, exist_ok=True)

# Copy classes.txt
shutil.copy(os.path.join(data_path, 'classes.txt'),
            os.path.join(base_output, 'classes.txt'))

# Get image list (filter only images)
img_file_list = list(Path(input_image_path).glob('*.*'))

print(f'Number of image files: {len(img_file_list)}')

# Shuffle once (important fix)
random.shuffle(img_file_list)

# Split
file_num = len(img_file_list)
train_num = int(file_num * train_percent)

train_files = img_file_list[:train_num]
val_files = img_file_list[train_num:]

print(f'Images moving to train: {len(train_files)}')
print(f'Images moving to validation: {len(val_files)}')

# Function to copy files
def copy_files(file_list, img_dest, txt_dest):
    for img_path in file_list:
        img_fn = img_path.name
        base_fn = img_path.stem
        txt_fn = base_fn + '.txt'
        txt_path = os.path.join(input_label_path, txt_fn)

        # Copy image
        shutil.copy(img_path, os.path.join(img_dest, img_fn))

        # Copy label if exists
        if os.path.exists(txt_path):
            shutil.copy(txt_path, os.path.join(txt_dest, txt_fn))

# Copy train and validation
copy_files(train_files, train_img_path, train_txt_path)
copy_files(val_files, val_img_path, val_txt_path)

print("✅ Dataset split completed!")

Number of image files: 162
Images moving to train: 129
Images moving to validation: 33
✅ Dataset split completed!


**install utitaries**

In [1]:
!pip install ultralytics

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.2/1.2 MB 19.6 MB/s eta 0:00:00a 0:00:01


**Configure training**

In [6]:
import yaml
import os

def create_data_yaml(path_to_classes_txt, path_to_data_yaml):

    # Read classes.txt
    if not os.path.exists(path_to_classes_txt):
        print(f'classes.txt file not found at {path_to_classes_txt}')
        return

    with open(path_to_classes_txt, 'r') as f:
        classes = [line.strip() for line in f if line.strip()]

    number_of_classes = len(classes)

    # Create data dictionary (Kaggle paths)
    data = {
        'path': '/kaggle/working/data',   # root folder
        'train': 'train/images',
        'val': 'validation/images',
        'nc': number_of_classes,
        'names': classes
    }

    # Write YAML
    with open(path_to_data_yaml, 'w') as f:
        yaml.dump(data, f, sort_keys=False)

    print(f'✅ Created config file at {path_to_data_yaml}')


# ===== Paths for Kaggle =====
path_to_classes_txt = '/kaggle/working/data/classes.txt'
path_to_data_yaml = '/kaggle/working/data.yaml'

# Run
create_data_yaml(path_to_classes_txt, path_to_data_yaml)

# Show file contents
with open(path_to_data_yaml, 'r') as f:
    print("\n📄 data.yaml contents:\n")
    print(f.read())

✅ Created config file at /kaggle/working/data.yaml

📄 data.yaml contents:

path: /kaggle/working/data
train: train/images
val: validation/images
nc: 11
names:
- MMs_peanut
- MMs_regular
- airheads
- gummy_worms
- milky_way
- nerds
- skittles
- snickers
- starbust
- three_musketeers
- twizzlers



In [13]:
!git clone https://github.com/WongKinYiu/yolov7.git

Cloning into 'yolov7'...
remote: Enumerating objects: 1197, done.
remote: Total 1197 (delta 0), reused 0 (delta 0), pack-reused 1197 (from 1)
Receiving objects: 100% (1197/1197), 74.29 MiB | 35.93 MiB/s, done.
Resolving deltas: 100% (511/511), done.


In [16]:
%cd /kaggle/working/yolov7


/kaggle/working/yolov7


In [20]:
!wget https://github.com/WongKinYiu/yolov7/releases/download/v0.1/yolov7.pt

--2026-03-17 16:05:24--  https://github.com/WongKinYiu/yolov7/releases/download/v0.1/yolov7.pt
Resolving github.com (github.com)... 140.82.113.4
Connecting to github.com (github.com)|140.82.113.4|:443... connected.
HTTP request sent, awaiting response... 302 Found
Location: https://release-assets.githubusercontent.com/github-production-release-asset/511187726/b0243edf-9fb0-4337-95e1-42555f1b37cf?sp=r&sv=2018-11-09&sr=b&spr=https&se=2026-03-17T16%3A39%3A31Z&rscd=attachment%3B+filename%3Dyolov7.pt&rsct=application%2Foctet-stream&skoid=96c2d410-5711-43a1-aedd-ab1947aa7ab0&sktid=398a6654-997b-47e9-b12b-9515b896b4de&skt=2026-03-17T15%3A38%3A58Z&ske=2026-03-17T16%3A39%3A31Z&sks=b&skv=2018-11-09&sig=TJgfwWveiY8HaO38oH%2B60pBqM83FbF2JfiebX%2FOM04g%3D&jwt=eyJ0eXAiOiJKV1QiLCJhbGciOiJIUzI1NiJ9.eyJpc3MiOiJnaXRodWIuY29tIiwiYXVkIjoicmVsZWFzZS1hc3NldHMuZ2l0aHVidXNlcmNvbnRlbnQuY29tIiwia2V5Ijoia2V5MSIsImV4cCI6MTc3Mzc2NTMyNCwibmJmIjoxNzczNzYzNTI0LCJwYXRoIjoicmVsZWFzZWFzc2V0cHJvZHVjdGlvbi5ibG9iLmNvcmUud2

**GIoU Loss**

In [ ]:
def giou_loss(pred_boxes, target_boxes):
    """
    pred_boxes, target_boxes: (N, 4) in (x1, y1, x2, y2) format
    """
    x1p, y1p, x2p, y2p = pred_boxes.unbind(-1)
    x1t, y1t, x2t, y2t = target_boxes.unbind(-1)
    
    # Intersection
    x1i = torch.max(x1p, x1t)
    y1i = torch.max(y1p, y1t)
    x2i = torch.min(x2p, x2t)
    y2i = torch.min(y2p, y2t)
    inter_area = (x2i - x1i).clamp(0) * (y2i - y1i).clamp(0)
    
    # Union
    area_p = (x2p - x1p) * (y2p - y1p)
    area_t = (x2t - x1t) * (y2t - y1t)
    union_area = area_p + area_t - inter_area
    
    iou = inter_area / (union_area + 1e-7)
    
    # Enclosing box
    x1c = torch.min(x1p, x1t)
    y1c = torch.min(y1p, y1t)
    x2c = torch.max(x2p, x2t)
    y2c = torch.max(y2p, y2t)
    area_c = (x2c - x1c) * (y2c - y1c)
    
    giou = iou - (area_c - union_area) / (area_c + 1e-7)
    loss = 1 - giou
    return loss.mean()